In [1]:
import os
os.chdir('../')

In [2]:
%pwd


'c:\\Users\\VENKATESH\\ML_project'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen = True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [4]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifact_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema
        )

        return data_validation_config


In [6]:
import os
from mlProject import logger


In [7]:
import pandas as pd


class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_schema(self) -> bool:
        try:
            validation_status = True

            data = pd.read_csv(self.config.unzip_data_dir)
            schema = self.config.all_schema

            for column, expected_dtype in schema.items():
                if column not in data.columns:
                    validation_status = False
                    break

                actual_dtype = str(data[column].dtype)

                if actual_dtype != expected_dtype:
                    validation_status = False
                    break

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e


In [8]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config = data_validation_config)
    data_validation.validate_schema()

except Exception as e:
    raise e


[2026-05-06 13:49:01,608: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-05-06 13:49:01,610: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-05-06 13:49:01,614: INFO: common: YAML file loaded successfully from: schema.yaml]
[2026-05-06 13:49:01,615: INFO: common: Created directory at: artifacts]
[2026-05-06 13:49:01,617: INFO: common: Created directory at: artifacts/data_validation]
